# Embedding contestuali: la parola «pesca» cambia vettore col contesto (v3)

In questa demo verifichiamo con i numeri quanto visto nelle slide: **lo stesso token scritto riceve vettori diversi a seconda della frase**. Usiamo la parola **«pesca»**, identica nello scritto per due significati (il *frutto* e l'*attività*): per il modello l'unico indizio è il contesto.

Novità della v3, nate dalle prove precedenti: modello addestrato solo sull'italiano, **8 frasi** sintatticamente riequilibrate (4 per senso), lettura dal **penultimo strato**, e soprattutto il confronto tra la similarità **grezza** (che mostreremo per prima, con il suo difetto) e quella **dopo centratura** (la correzione).

Termini di base che useremo:
- **embedding contestuale**: il vettore che l'encoder produce per un token *dopo* aver guardato tutta la frase;
- **sotto-token (subword)**: il tokenizzatore può spezzare una parola in pezzi (es. `pe` + `##sca`); in quel caso prendiamo la **media** dei vettori dei pezzi;
- **similarità coseno**: misura dell'angolo tra due vettori; 1 = stessa direzione, valori più bassi = direzioni diverse.


## Note sull'ambiente e avvisi attesi

**Perché per ora giriamo su CPU.** La workstation monta una RTX 5090 (architettura Blackwell, *compute capability* sm_120 — la versione dell'architettura CUDA per cui i kernel della libreria, cioè le sue funzioni di calcolo precompilate, devono essere stati compilati). La build di PyTorch oggi nell'ambiente è una `+cu126`, che arriva fino a sm_90: la GPU viene vista dal driver, ma alla prima computazione mancherebbero i kernel. Per questa demo la CPU basta e avanza, quindi nella cella di preparazione fissiamo `device = "cpu"`.

**Quando vorremo la GPU.** Nell'ambiente attivo reinstalliamo torch nella variante giusta e verifichiamo:

```
python -m pip install --force-reinstall torch --index-url https://download.pytorch.org/whl/cu128
python -c "import torch; print(torch.__version__, torch.cuda.get_arch_list())"
```

Nell'elenco deve comparire `sm_120`; a quel punto ripristiniamo la riga automatica `device = "cuda" if torch.cuda.is_available() else "cpu"`.




In [1]:
# Installazioni, secondo la nostra strategia: transformers e pandas via conda
# (canale conda-forge, con --override-channels); torch via python -m pip.

import torch
from transformers import AutoTokenizer, AutoModel

MODELLO = "dbmdz/bert-base-italian-xxl-cased"   # BERT addestrato solo su corpora italiani

# Per ora lavoriamo su CPU (vedi "Note sull'ambiente" in alto).
# Dopo il reinstall di torch in variante cu128, ripristiniamo la riga automatica:
# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODELLO)
model = AutoModel.from_pretrained(MODELLO).to(device).eval()
print(f"Modello caricato su: {device}")


config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-italian-xxl-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modello caricato su: cpu


**Avvisi attesi alla prima esecuzione (innocui).**
- Il *load report* con righe `UNEXPECTED` su `cls.*`: sono le teste di pre-addestramento di BERT — MLM (*masked language modeling*: predire le parole mascherate) e, se presenti, NSP (*next sentence prediction*) — che non carichiamo perché usiamo solo il tronco `BertModel`; quei pesi vengono scartati.
- `TqdmWarning: IProgress not found`: puramente estetico (barre di avanzamento grafiche in Jupyter); si elimina con `conda install -c conda-forge --override-channels ipywidgets` nell'ambiente del kernel, seguito dal riavvio del kernel.
- Il modello di questa versione è nuovo: al primo avvio viene scaricato (circa mezzo GB), poi resta in cache.

## La funzione che estrae il vettore di una parola

Come nelle versioni precedenti: troviamo la parola nella frase, tokenizziamo chiedendo gli *offset* (a quali caratteri corrisponde ogni token), passiamo la frase nel modello e facciamo la media dei token che coprono la parola.

Una differenza: chiediamo al modello **tutti gli strati** (`output_hidden_states=True`) e leggiamo il **penultimo** (`STRATO = -2`). L'ultimo strato è "tirato" verso l'obiettivo di pre-addestramento (predire parole mascherate), mentre il senso lessicale è rappresentato meglio un gradino sotto. `hidden_states` è la lista degli stati nascosti: l'indice 0 è lo strato di embedding iniziale, −1 l'ultimo, −2 il penultimo — ed è un parametro con cui si può sperimentare.


In [2]:
STRATO = -2   # penultimo strato; provare anche -1 (ultimo) o valori intermedi

def vettore_parola(frase, parola):
    """Embedding contestuale (strato STRATO) di `parola` dentro `frase`.
    Se la parola è spezzata in più sotto-token, restituiamo la media dei loro vettori."""
    inizio = frase.lower().find(parola.lower())
    assert inizio != -1, f"'{parola}' non trovata nella frase: {frase}"
    fine = inizio + len(parola)

    enc = tokenizer(frase, return_tensors="pt", return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")[0].tolist()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        uscita = model(**enc, output_hidden_states=True)
    hidden = uscita.hidden_states[STRATO][0]   # un vettore per ogni token della frase

    indici = [i for i, (a, b) in enumerate(offsets)
              if a < fine and b > inizio and b > a]    # token che coprono la parola
    return hidden[indici].mean(dim=0)


In [3]:
frasi = {
    "frutto 1":    "La pesca matura colta dall'albero è dolcissima.",
    "frutto 2":    "La pesca comprata al mercato è succosa e profumata.",
    "frutto 3":    "La pesca va sbucciata prima di essere mangiata.",
    "frutto 4":    "La pesca ha la buccia vellutata e il nocciolo duro.",
    "attivita' 1": "La pesca con la canna sul lago richiede pazienza.",
    "attivita' 2": "La pesca notturna in alto mare è vietata senza licenza.",
    "attivita' 3": "La pesca a strascico danneggia i fondali marini.",
    "attivita' 4": "La pesca sportiva sul fiume è regolata da permessi.",
}

vettori = {nome: vettore_parola(frase, "pesca") for nome, frase in frasi.items()}
nomi = list(vettori)
print("Occorrenze:", len(nomi), "| dimensione di ogni vettore:", tuple(vettori[nomi[0]].shape))


Occorrenze: 8 | dimensione di ogni vettore: (768,)


## Prima misura: la similarità grezza (e il suo difetto)

Partiamo dal metro usato finora, così com'è. Due funzioni di servizio: una calcola in un colpo solo la similarità coseno tra tutte le coppie, l'altra mostra la matrice in forma **triangolare inferiore** (la matrice è simmetrica: la metà sopra la diagonale ripeterebbe quella sotto; la diagonale vale sempre 1).

Il difetto da osservare nei numeri: gli spazi di questi modelli sono **anisotropi**, cioè i vettori non si distribuiscono in tutte le direzioni ma si concentrano in un "cono" ristretto. Risultato: le similarità grezze escono tutte alte e ravvicinate, e i divari tra sensi diversi restano di pochi centesimi — troppo poco per un ordinamento affidabile.


In [4]:
import pandas as pd
import torch.nn.functional as F

def matrice_cos(M):
    """Similarità coseno tra tutte le coppie di righe di M (confronto in blocco via broadcasting)."""
    return F.cosine_similarity(M.unsqueeze(1), M.unsqueeze(0), dim=-1).tolist()

def triangolare(sim, nomi):
    """Matrice come DataFrame di stringhe: solo triangolare inferiore, diagonale inclusa."""
    tri = pd.DataFrame("", index=nomi, columns=nomi)
    for i in range(len(nomi)):
        for j in range(i + 1):
            tri.iloc[i, j] = f"{sim[i][j]:.3f}"
    return tri

M = torch.stack([vettori[n] for n in nomi])
sim_grezza = matrice_cos(M)
triangolare(sim_grezza, nomi)


,frutto 1,frutto 2,frutto 3,frutto 4,attivita' 1,attivita' 2,attivita' 3,attivita' 4
frutto 1,1.000,,,,,,,
frutto 2,0.924,1.000,,,,,,
frutto 3,0.879,0.905,1.000,,,,,
frutto 4,0.839,0.872,0.898,1.000,,,,
attivita' 1,0.628,0.659,0.659,0.667,1.000,,,
attivita' 2,0.619,0.637,0.631,0.610,0.856,1.000,,
attivita' 3,0.673,0.679,0.699,0.689,0.845,0.816,1.000,
attivita' 4,0.656,0.679,0.663,0.639,0.849,0.910,0.783,1.000


## La correzione: centratura

Ogni vettore si può pensare come *parte comune a tutti* (il cono dell'anisotropia) più *parte distintiva*. La similarità grezza le misura insieme; a noi interessa solo la seconda. La **centratura** fa esattamente questo: calcoliamo il **vettore medio** delle 8 occorrenze e lo sottraiamo da ciascuna, poi rimisuriamo la coseno sui vettori centrati.

La scala ora si riapre: compaiono valori bassi e anche **negativi**, che si leggono come "dai lati opposti rispetto al centro". È qui che i due blocchi devono separarsi.


In [5]:
Mc = M - M.mean(dim=0, keepdim=True)   # da ogni vettore togliamo il vettore medio
sim_centrata = matrice_cos(Mc)
triangolare(sim_centrata, nomi)


,frutto 1,frutto 2,frutto 3,frutto 4,attivita' 1,attivita' 2,attivita' 3,attivita' 4
frutto 1,1.000,,,,,,,
frutto 2,0.599,1.000,,,,,,
frutto 3,0.390,0.457,1.000,,,,,
frutto 4,0.231,0.329,0.482,1.000,,,,
attivita' 1,-0.638,-0.618,-0.618,-0.464,1.000,,,
attivita' 2,-0.600,-0.633,-0.666,-0.624,0.506,1.000,,
attivita' 3,-0.477,-0.561,-0.467,-0.401,0.418,0.342,1.000,
attivita' 4,-0.529,-0.540,-0.608,-0.595,0.438,0.683,0.167,1.000


## Il verdetto in due numeri

Per non affidarci all'occhio, riassumiamo ogni matrice con due medie: quella delle coppie **intra-senso** (frutto–frutto e attività–attività) e quella delle coppie **inter-senso** (frutto–attività). La demo "funziona" quando il divario intra − inter è netto.


In [ ]:
def medie_intra_inter(sim, nomi):
    intra, inter = [], []
    for i in range(len(nomi)):
        for j in range(i):
            stesso_senso = nomi[i].split()[0] == nomi[j].split()[0]
            (intra if stesso_senso else inter).append(sim[i][j])
    return sum(intra) / len(intra), sum(inter) / len(inter)

for etichetta, sim in [("grezza   ", sim_grezza), ("centrata ", sim_centrata)]:
    intra, inter = medie_intra_inter(sim, nomi)
    print(f"similarità {etichetta}: intra = {intra:+.3f} | inter = {inter:+.3f} | divario = {intra - inter:+.3f}")


## Come leggere i risultati (e perché prima "non funzionava")

Sulla matrice **grezza** ci aspettiamo il quadro già visto: valori tutti alti e vicini, ordinamenti fragili. Sulla matrice **centrata** ci aspettiamo che le coppie intra-senso restino chiaramente sopra le inter-senso, con un divario ben visibile anche nelle due medie finali. I valori esatti dipendono da modello e strato: a contare è il divario, non i numeri assoluti.

Il riepilogo delle cause viste strada facendo, buono anche per l'aula:
1. **anisotropia**: la coseno grezza misura anche la parte comune a tutti i vettori, e la scala si schiaccia — la centratura la rimuove;
2. **sintassi come variabile di disturbo**: costruzioni diverse ("a pesca" avverbiale contro "la pesca" nominale) spostano i vettori quanto il senso — frasi riequilibrate isolano il significato;
3. **modello multilingue**: dedica all'italiano una fetta modesta della sua capacità — un modello solo-italiano separa meglio;
4. **ultimo strato**: orientato al compito di pre-addestramento — il penultimo rappresenta meglio il senso lessicale.

Varianti da provare: cambiare `STRATO` (da −1 agli strati intermedi) e osservare il divario; altre parole ambigue («calcio», «riso», «vite», «piano»); una frase «neutra» («La pesca è davvero bella.») da aggiungere e collocare.
